In [4]:
import MySQLdb

In [5]:
# 데이터 모델
class Contact:
    def __init__(self, name, phone, memo="없음"):
        self.name = name
        self.phone = phone
        self.memo = memo

    def __repr__(self):
        return f"Contact(name={self.name}, phone={self.phone}, memo={self.memo})"
    
    @property
    def name(self):
        return self.__name
    
    @name.setter
    def name(self, name):
        if not name:
            raise ValueError("이름은 비워둘 수 없습니다")
        self.__name = name

    @property
    def phone(self):
        return self.__phone
    
    @phone.setter
    def phone(self, phone):
        if not phone:
            raise ValueError("연락처는 비워둘 수 없습니다")
        self.__phone = phone

    @property
    def memo(self):
        return self.__memo
    
    @memo.setter
    def memo(self, memo):
        self.__memo = memo
    

In [ ]:
# DB 접근 객체
class ContactsDAO:
    def __init__(self):
        self.db = None
        
    def connect(self):
        if not MySQLdb:
            raise ImportError("python -m pip install mysqlclient")

        self.db = MySQLdb.connect(host='localhost', user='root', password='1234', db='contact', charset='utf8')

    def disconnect(self):
        if self.db:
            self.db.close()

    def insert(self, contact):
        self.connect()
        cur = self.db.cursor()

        sql = "insert into contact (name, phone, memo) values (%s, %s, %s)"
        data = (contact.name, contact.phone, contact.memo)
        cur.execute(sql, data)
        self.db.commit()
        cur.close()
        self.disconnect()
    
    def select_all(self):
        self.connect()
        cur = self.db.cursor(MySQLdb.cursors.DictCursor)

        sql = "select name, phone, memo from contact order by name"
        cur.execute(sql)
        rows = cur.fetchall()
        cur.close()
        self.disconnect()
        return rows
    
    def search_all(self, name):
        self.connect()
        cur = self.db.cursor(MySQLdb.cursors.DictCursor)

        sql = "select name, phone, memo from contact where name like concat('%%', %s, '%%')"
        cur.execute(sql, (name,))
        rows = cur.fetchall()
        cur.close()
        self.disconnect()
        return rows
    
    def search(self, name):
        self.connect()
        cur = self.db.cursor(MySQLdb.cursors.DictCursor)

        sql = "select name, phone, memo from contact where name=%s"
        cur.execute(sql, (name,))
        row = cur.fetchone()
        cur.close()
        self.disconnect()
        return row

    def update(self, contact):
        self.connect()
        cur = self.db.cursor()

        sql = "update contact set phone=%s, memo=%s where name=%s"
        data = (contact.phone, contact.memo, contact.name)
        result = cur.execute(sql, data)
        self.db.commit()
        cur.close()
        self.disconnect()
        return result
    
    def delete(self, name):
        self.connect()
        cur = self.db.cursor()

        sql = "delete from contact where name=%s"
        result = cur.execute(sql, (name,))
        self.db.commit()
        cur.close()
        self.disconnect()
        return result

In [ ]:
# 비즈니스 로직
class ContactsService:
    def __init__(self):
        self.dao = ContactsDAO()
        
    def insert_contact(self):
        name = input("이름을 입력하세요: ")
        phone = input("연락처를 입력하세요: ")
        memo = input("메모를 입력하세요: ")

        contact = Contact(name, phone, memo)
        self.dao.insert(contact)
        print("연락처가 등록되었습니다")

    def print_all(self):
        contacts = self.dao.select_all()
        if not contacts:
            print("등록된 연락처가 없습니다")
            return
        for contact in contacts:
            print(f"{contact['name']} {contact['phone']}, 메모: {contact['memo']}")

    def search_contacts(self):
        name = input("연락처를 검색할 이름을 입력하세요: ")
        contacts = self.dao.search_all(name)
        if not contacts:
            print("등록된 연락처가 없습니다")
            return
        for contact in contacts:
            print(f"{contact['name']} {contact['phone']}, 메모: {contact['memo']}")

    def edit_contact(self):
        name = input("연락처를 수정할 이름을 입력하세요: ")
        contact = self.dao.search(name)
        if not contact:
            print("수정할 연락처가 없습니다")
            return
        phone = input("새로운 연락처를 입력하세요: ")
        memo = input("새로운 메모를 입력하세요: ")
        contact = Contact(name, phone, memo)
        result = self.dao.update(contact)
        if result > 0:
            print("연락처가 수정되었습니다")
        else:
            print("연락처 수정에 실패했습니다")

    def delete_contact(self):
        name = input("연락처를 삭제할 이름을 입력하세요: ")
        contact = self.dao.search(name)
        if not contact:
            print("삭제할 연락처가 없습니다")
            return
        result = self.dao.delete(name)
        if result > 0:
            print("연락처가 삭제되었습니다")
        else:
            print("연락처 삭제에 실패했습니다")


In [ ]:
# 사용자 인터페이스
class Menu:
    def __init__(self):
        self.service = ContactsService()

    def run(self):
        while True:
            try:
                print()
                print("=======연락처=======")
                print("1. 연락처 등록하기")
                print("2. 전체 연락처 출력하기")
                print("3. 연락처 검색하기")
                print("4. 연락처 수정하기")
                print("5. 연락처 삭제하기")
                print("6. 연락처 종료하기")
                menu = int(input("메뉴를 선택하세요"))

                if menu == 1: 
                    print("연락처를 등록합니다")
                    self.service.insert_contact()
                elif menu == 2:
                    print("전체 연락처를 출력합니다")
                    self.service.print_all()
                elif menu == 3:
                    print("연락처를 검색합니다")
                    self.service.search_contacts()
                elif menu == 4:
                    print("연락처를 수정합니다")
                    self.service.edit_contact()
                elif menu == 5:
                    print("연락처를 삭제합니다")
                    self.service.delete_contact()
                elif menu == 6:
                    print("프로그램을 종료합니다")
                    break
                else: 
                    print("메뉴는 1부터 6까지 입력해주세요")

            except Exception as e:
                print("오류: ", e)
                print("다시 입력하세요")

In [10]:
# 연락처 실행
menu = Menu()
menu.run()


=======연락처=======
1. 연락처 등록하기
2. 전체 연락처 출력하기
3. 연락처 검색하기
4. 연락처 수정하기
5. 연락처 삭제하기
6. 연락처 종료하기
전체 연락처를 출력합니다
오류:  (1049, "Unknown database 'contact'")
다시 입력하세요

=======연락처=======
1. 연락처 등록하기
2. 전체 연락처 출력하기
3. 연락처 검색하기
4. 연락처 수정하기
5. 연락처 삭제하기
6. 연락처 종료하기
연락처를 등록합니다
오류:  (1049, "Unknown database 'contact'")
다시 입력하세요

=======연락처=======
1. 연락처 등록하기
2. 전체 연락처 출력하기
3. 연락처 검색하기
4. 연락처 수정하기
5. 연락처 삭제하기
6. 연락처 종료하기


KeyboardInterrupt: Interrupted by user